
# Обучение модели YOLOv8 для Raspberry Pi Zero 2 W (Bookworm 32-bit) + CSI
Датасет готовится и экспортируется в **Roboflow** в формате **YOLOv8**.

> Рекомендация для дальнейшего запуска на Raspberry Pi Zero 2 W: инференс лучше делать через TFLite, а не PyTorch.


## Установка зависимостей (PC)

In [1]:
# Выполни один раз (если ещё не установлено):
# pip install ultralytics opencv-python pyyaml matplotlib

import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

Python: 3.12.10
Platform: macOS-15.7.1-arm64-arm-64bit


## Импорты

In [2]:
from ultralytics import YOLO
import cv2
import yaml
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Ultralytics OK")

Ultralytics OK


## Экспорт датасета из Roboflow (как подготовить)

- Создайте папку `dataset/`
- Распакуйте Roboflow zip **внутрь** `dataset/`

## Укажите путь к датасету Roboflow

In [3]:

# Укажите путь к папке, где лежит data.yaml
DATA_ROOT = Path("./dataset").resolve()

data_yaml_path = DATA_ROOT / "data.yaml"
print("DATA_ROOT =", DATA_ROOT)
print("data.yaml =", data_yaml_path)

assert DATA_ROOT.exists(), f"DATA_ROOT не найден: {DATA_ROOT}"
assert data_yaml_path.exists(), f"Не найден data.yaml: {data_yaml_path}"


DATA_ROOT = /Users/aleksioprime/edu/gymnasium/sk_aisb/training/dataset
data.yaml = /Users/aleksioprime/edu/gymnasium/sk_aisb/training/dataset/data.yaml


AssertionError: Не найден data.yaml: /Users/aleksioprime/edu/gymnasium/sk_aisb/training/dataset/data.yaml

## Приведение путей в `data.yaml` к абсолютным

In [ ]:

data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
print("data.yaml keys:", data.keys())

# Roboflow обычно кладёт train/val(valid)/test относительными путями.
# Ultralytics нормально работает с абсолютным path, поэтому нормализуем.

# Определим базовый path
base_path = Path(data.get("path", str(DATA_ROOT)))
if not base_path.is_absolute():
    base_path = (DATA_ROOT / base_path).resolve()

def normalize_split(v):
    p = Path(v)
    if p.is_absolute():
        return str(p)
    return str((base_path / p).resolve())

# Согласуем val/valid
if "val" not in data and "valid" in data:
    data["val"] = data["valid"]

assert "train" in data, "В data.yaml нет ключа train"
assert "val" in data, "В data.yaml нет ключа val (или valid)"

data["path"] = str(base_path)
data["train"] = normalize_split(data["train"])
data["val"] = normalize_split(data["val"])
if "test" in data:
    data["test"] = normalize_split(data["test"])

# Имена классов
names = data.get("names")
if isinstance(names, dict):
    # {0: '...', 1:'...'}
    CLASS_NAMES = [names[i] for i in sorted(names.keys())]
elif isinstance(names, list):
    CLASS_NAMES = names
else:
    raise ValueError("Не найдены names в data.yaml (ожидаем список или словарь)")

print("Base path:", data["path"])
print("Train:", data["train"])
print("Val:", data["val"])
print("Classes:", CLASS_NAMES)

# Сохраним нормализованный yaml рядом (чтобы точно работало из любой папки)
normalized_yaml_path = Path("data_normalized.yaml").resolve()
normalized_yaml_path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("Saved normalized yaml:", normalized_yaml_path)


## 6) Санити-чек: показать несколько train изображений с боксами

In [ ]:

# Roboflow структура: train/images, train/labels и valid/images, valid/labels
train_images_dir = Path(data["train"]) / "images"
train_labels_dir = Path(data["train"]) / "labels"

assert train_images_dir.exists(), f"Не найдена папка train images: {train_images_dir}"
assert train_labels_dir.exists(), f"Не найдена папка train labels: {train_labels_dir}"

def read_yolo_labels(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes
    txt = label_path.read_text(encoding="utf-8").strip()
    if not txt:
        return boxes
    for line in txt.splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls, xc, yc, w, h = parts
        boxes.append((int(cls), float(xc), float(yc), float(w), float(h)))
    return boxes

def draw_boxes(img_bgr, boxes, class_names):
    h, w = img_bgr.shape[:2]
    out = img_bgr.copy()
    for cls, xc, yc, bw, bh in boxes:
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w-1, x2), min(h-1, y2)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0,255,0), 2)
        name = class_names[cls] if 0 <= cls < len(class_names) else str(cls)
        cv2.putText(out, name, (x1, max(0, y1-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    return out

imgs = sorted(list(train_images_dir.glob("*.*")))
assert len(imgs) > 0, f"Нет изображений в {train_images_dir}"

for img_path in random.sample(imgs, k=min(3, len(imgs))):
    label_path = train_labels_dir / (img_path.stem + ".txt")
    img = cv2.imread(str(img_path))
    assert img is not None, f"Не удалось прочитать: {img_path}"
    boxes = read_yolo_labels(label_path)
    vis = draw_boxes(img, boxes, CLASS_NAMES)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(9, 6))
    plt.title(img_path.name)
    plt.axis("off")
    plt.imshow(vis_rgb)
    plt.show()


## 7) Обучение YOLOv8n (на PC)

In [ ]:

MODEL_NAME = "yolov8n.pt"   # nano
EPOCHS = 80
IMGSZ_TRAIN = 640          # 640 для качества; если хочешь edge-ориентированно — попробуй 416/320
BATCH = 16                 # на CPU часто нужно 4-8; на Apple Silicon MPS можно 8-16
WORKERS = 2

# device:
# - macOS Apple Silicon: "mps" (если доступно)
# - Windows/NVIDIA: 0
# - CPU: "cpu"
DEVICE = "cpu"

# Попробуем автоматически выбрать MPS на macOS (Apple Silicon), если доступно
try:
    import torch
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        DEVICE = "mps"
except Exception:
    pass

print("Using DEVICE =", DEVICE)

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(normalized_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ_TRAIN,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project="runs_train",
    name="exp_pc_roboflow",
    patience=20,
    close_mosaic=10,
)
print("Train done.")


## 8) Валидация + путь к лучшим весам

In [ ]:

best_pt = Path("runs_train/exp_pc_roboflow/weights/best.pt")
last_pt = Path("runs_train/exp_pc_roboflow/weights/last.pt")

print("best:", best_pt, "exists:", best_pt.exists())
print("last:", last_pt, "exists:", last_pt.exists())
assert best_pt.exists(), "Не найден best.pt — проверь, что обучение завершилось."

trained = YOLO(str(best_pt))

metrics = trained.val(
    data=str(normalized_yaml_path),
    imgsz=IMGSZ_TRAIN,
    device=DEVICE,
)
print(metrics)


## 9) Быстрый predict на val и сохранение результатов

In [ ]:

val_dir = Path(data["val"]) / "images"
assert val_dir.exists(), f"Не найдена папка val images: {val_dir}"

_ = trained.predict(
    source=str(val_dir),
    imgsz=320,       # ближе к будущему edge-инференсу
    conf=0.35,
    device=DEVICE,
    save=True,
    project="runs_predict",
    name="val_pred_pc",
    verbose=False
)
print("Saved predictions to runs_predict/val_pred_pc")


## 10) Экспорт для Raspberry (TFLite FP16 + попытка INT8)

In [ ]:

# FP16 TFLite (обычно самый простой старт)
tflite_fp16_path = trained.export(format="tflite", imgsz=320, half=True)
print("TFLite FP16 exported:", tflite_fp16_path)



### Экспорт TFLite INT8 (калибровка)

INT8 может заметно ускорить инференс на Raspberry, но требует representative dataset (калибровки).
Мы используем часть train/images для калибровки.


In [ ]:

CALIB_IMAGES = 200
IMGSZ_EXPORT = 320

calib_imgs = sorted(list(train_images_dir.glob("*.*")))[:min(CALIB_IMAGES, len(list(train_images_dir.glob('*.*'))))]
assert len(calib_imgs) > 0, "Нет изображений для калибровки"

def representative_data_gen():
    for p in calib_imgs:
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMGSZ_EXPORT, IMGSZ_EXPORT), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)
        yield [img]

# В зависимости от версии ultralytics, INT8 экспорт может использовать data.yaml и внутренний rep-dataset.
# Если INT8 экспорт упадёт, попробуй уменьшить CALIB_IMAGES и/или поставить обучение/экспорт в Colab.
try:
    tflite_int8_path = trained.export(format="tflite", imgsz=IMGSZ_EXPORT, int8=True, data=str(normalized_yaml_path))
    print("TFLite INT8 exported:", tflite_int8_path)
except Exception as e:
    print("INT8 export failed:", repr(e))
    print("Рекомендация: попробуй Colab для INT8 экспорта или обнови ultralytics/onnx/tensorflow зависимости.")


## 11) Упаковка артефактов в zip

In [ ]:

import zipfile

artifacts = []
if best_pt.exists():
    artifacts.append(best_pt)
if normalized_yaml_path.exists():
    artifacts.append(normalized_yaml_path)

# tflite файлы
for root in [Path("."), Path("runs_train"), Path("runs_predict")]:
    if root.exists():
        artifacts.extend(root.rglob("*.tflite"))

# пару картинок из предикта
pred_dir = Path("runs_predict/val_pred_pc")
if pred_dir.exists():
    imgs = list(pred_dir.rglob("*.jpg")) + list(pred_dir.rglob("*.png"))
    artifacts.extend(imgs[:10])

zip_path = Path("artifacts_pc_roboflow.zip").resolve()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:
        p = Path(p)
        if p.exists() and p.is_file():
            z.write(p, arcname=str(p))

print("Created:", zip_path)
print("Included files:", sum(1 for p in artifacts if Path(p).exists()))
